In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import math

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


In [ ]:
import torch
import torch.nn as nn
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import cv2

class OutpaintGenerator:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")
        self.scheduler = DDPMScheduler.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            subfolder="scheduler"
        )

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()

        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj]
        self.vae, self.unet, self.coord_encoder, self.cond_proj = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr = 1e-5)

        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)
        # Prepare for inference
        self.vae.requires_grad_(False)
        self.unet.requires_grad_(False)
        self.coord_encoder.requires_grad_(False)
        self.cond_proj.requires_grad_(False)

        # Define transformation
        self.transform = transforms.Compose([
            #transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])

    def _create_latent_mask(self, bbox, latent_shape):
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            # Convert coordinates to latent space
            x1 = coords[0] * lw
            y1 = coords[1] * lh
            x2 = coords[2] * lw
            y2 = coords[3] * lh

            # Create basic mask
            xx, yy = torch.meshgrid(
                torch.arange(lw, device=self.accelerator.device),
                torch.arange(lh, device=self.accelerator.device))
            mask = ((xx >= x1) & (xx <= x2) & (yy >= y1) & (yy <= y2)).float()
            masks.append(mask)
        return torch.stack(masks).unsqueeze(1)

    def generate_image(self, masked_img, bbox, steps, n):
        # Encode masked image to latent space
        masked_latents = self.vae.encode(masked_img).latent_dist.sample()
        masked_latents = masked_latents * self.vae.config.scaling_factor
        coord_features = self.coord_encoder(bbox)
        condition = torch.cat([
            self.cond_proj(masked_latents),
            coord_features.unsqueeze(1).expand(-1, 64, -1)
        ], dim=-1)
        print(torch.mean(condition))

        # Create latent mask
        latent_masks = self._create_latent_mask(bbox, masked_latents.shape)
        #print(masked_latents.shape)

        # Add noise to latents based on the mask
        noise = torch.randn_like(masked_latents)
        #timesteps = torch.randint(0, self.noise_scheduler.config.num_train_timesteps, (masked_latents.shape[0],),
        #                          device=masked_latents.device)
        noisy_latents = self.noise_scheduler.add_noise(masked_latents * latent_masks, noise * latent_masks, torch.tensor(steps))
        noisy_latents = masked_latents * (1 - latent_masks) + noisy_latents * latent_masks
        latent_input = noisy_latents

        self.scheduler.set_timesteps(steps)

        for i, t in enumerate(self.scheduler.timesteps):
            latent_input = latent_input * latent_masks + masked_latents * (1-latent_masks)

            with torch.no_grad():
                noise_pred = self.unet(
                    latent_input, t,
                    encoder_hidden_states=condition
                ).sample

            # 8d. Scheduler Step
            # Compute de-noised sample using scheduler's update rule
            latent_input = self.scheduler.step(
                noise_pred, t, latent_input
            ).prev_sample

            # 8f. Intermediate Visualization
            # Display progress at specified intervals
            if i == len(self.scheduler.timesteps) - 1:
                with torch.no_grad():
                    # Decode blended latent while maintaining original regions
                    blended_latent = masked_latents * (1 - latent_masks) + latent_input * latent_masks
                    generated = self.vae.decode(blended_latent / self.vae.config.scaling_factor).sample
                    generated = generated[0].permute(1, 2, 0).cpu().numpy()
                    generated = (generated * 0.5 + 0.5).clip(0, 1)
                    masked_img1 = masked_img[0].cpu().permute(1, 2, 0).numpy()
                    masked_img1 = (masked_img1 * 0.5 + 0.5).clip(0, 1)
                    generated = (generated * 255).astype(np.uint8)
                    Image.fromarray(generated).save(f"{j}.png")

                    pad = 5

                    # 从 bbox 得到像素坐标
                    bbox_np = bbox[0].cpu().numpy() * 512
                    x1_rel, y1_rel, x2_rel, y2_rel = bbox_np
                    print(bbox_np)

                    x_min = int(y1_rel)
                    y_min = int(x1_rel)
                    x_max = int(y2_rel)
                    y_max = int(x2_rel)

                    # 转 float
                    generated_float = generated.astype(np.float32)
                    masked_img1_uint8 = (masked_img1 * 255).astype(np.uint8)
                    original_float = masked_img1_uint8.astype(np.float32)

                    # 修 mask 左边条
                    fix_margin = 1
                    patch = generated_float[y_min:y_max+1, x_min - fix_margin : x_min, :]
                    generated_float[y_min:y_max+1, x_min : x_min + fix_margin, :] = patch

                    patch = generated_float[y_min - fix_margin : y_min, x_min:x_max+1, :]
                    generated_float[y_min : y_min + fix_margin, x_min:x_max+1, :] = patch
                    # scale 回 uint8
                    generated_fixed = np.clip(generated_float, 0, 255).astype(np.uint8)

                    Image.fromarray(generated_fixed).save(f"drive/MyDrive/merfishinpaint/{n}.png")


                    # 可视化
                    plt.figure(figsize=(12, 6))
                    plt.subplot(131)
                    plt.imshow(masked_img1)
                    plt.title("Masked Input")
                    plt.axis('off')

                    plt.subplot(132)
                    plt.imshow(generated)
                    plt.title("Generated (raw)")
                    plt.axis('off')

                    plt.subplot(133)
                    plt.imshow(generated_fixed)
                    plt.title("Fixed Output (Bilinear Blend)")
                    plt.axis('off')
                    plt.show()

In [ ]:
generator = OutpaintGenerator()
generator.accelerator.load_state("drive/MyDrive/checkpoint_LR_merfish")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [ ]:
generator.unet.to("cuda:0")
generator.coord_encoder.to("cuda:0")
generator.cond_proj.to("cuda:0")
# Generate an image
for i in range(0,10):
    # Load a masked image and bounding box for inference
    masked_img = Image.open("region_13.png").convert("RGB")#Image.open(f"drive/MyDrive/merfish/merfish_region_images/region_{i}.png").convert("RGB")
    masked_img = generator.transform(masked_img).unsqueeze(0).to(generator.accelerator.device)
    for j in range(3,6):
      bbox = torch.tensor([[0.36, 0.25, 0.36+0.05*j, 0.25+0.05*j]])  # Example bounding box
      bbox = bbox.half()
      bbox = bbox.to("cuda:0")
      mask = torch.ones_like(masked_img)
      h, w = masked_img.shape[2], masked_img.shape[3]
      x1 = int(bbox[0][0] * w)
      y1 = int(bbox[0][1] * h)
      x2 = int(bbox[0][2] * w)
      y2 = int(bbox[0][3] * h)
      mask[:, :, x1:x2, y1:y2] = 0
      # Apply mask and augmentation
      masked_img = masked_img * mask
      generator.generate_image(masked_img, bbox, 100, n=str(i)+"_"+str(j)+"_"+str(100))
      generator.generate_image(masked_img, bbox, 150, n=str(i)+"_"+str(j)+"_"+str(150))
      generator.generate_image(masked_img, bbox, 200, n=str(i)+"_"+str(j)+"_"+str(200))
      generator.generate_image(masked_img, bbox, 250, n=str(i)+"_"+str(j)+"_"+str(100))
      generator.generate_image(masked_img, bbox, 300, n=str(i)+"_"+str(j)+"_"+str(150))
      generator.generate_image(masked_img, bbox, 200, n=str(i)+"_"+str(j)+"_"+str(200))

Output hidden; open in https://colab.research.google.com to view.